In [2]:
"""
CAT 3 Evaluation Cell - Exact Matching
"""
import os, re, json, uuid, pathlib
from datetime import datetime
from pathlib import Path
import ast
import pandas as pd

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
outdir = pathlib.Path("results") / f"cat3_hybrid_{timestamp}"
outdir.mkdir(parents=True, exist_ok=True)


GT_DIR = Path("ground_truths") / "cat3_questions_table"
DIFF_DIR = outdir / "eval_diffs_cat3"
DIFF_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_ID_COLS = {"NCT", "Authors", "Year", "PMID"}

def _normalize_unicode(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    repl = {
        "â€“": "-", "–": "-", "—": "-",
        "â‰¥": ">=", "≥": ">=",
        "â‰¤": "<=", "≤": "<=",
        " ": " ",
        "\u200b": "",
        "’": "'", "“": '"', "”": '"',
    }
    for k, v in repl.items():
        s = s.replace(k, v)
    return " ".join(s.strip().split())

def _norm_boolish(x):
    if x is None:
        return None
    s = _normalize_unicode(str(x)).lower()
    if s in {"true","t","1","yes","y"}:
        return True
    if s in {"false","f","0","no","n"}:
        return False
    return None

def _is_listish_string(s: str) -> bool:
    s = s.strip()
    return (s.startswith("[") and s.endswith("]")) or ("," in s) or (";" in s)

def _norm_listish(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, (list, tuple, set)):
        items = list(x)
    else:
        s = _normalize_unicode(str(x)).strip()
        try:
            # JSON-ish list
            if s.startswith("[") and s.endswith("]"):
                parsed = ast.literal_eval(s)
                items = parsed if isinstance(parsed, (list, tuple, set)) else [s]
            else:
                # split on commas/semicolons/pipes
                parts = re.split(r"[;,|]", s)
                items = [p for p in parts]
        except Exception:
            items = [s]
    # normalize tokens
    norm = []
    for it in items:
        t = _normalize_unicode(str(it)).strip().lower()
        if t:
            norm.append(t)
    return sorted(set(norm))

def _norm_scalar(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    b = _norm_boolish(x)
    if b is not None:
        return str(b)
    s = _normalize_unicode(str(x)).strip().lower()
    if s in {"", "na", "n/a", "nr", "not reported", "--", "-"}:
        return ""
    return s

def _is_list_column(series: pd.Series) -> bool:
    sample = series.dropna().astype(str).head(5).tolist()
    return any(_is_listish_string(v) for v in sample)

def evaluate_pair(gt: pd.DataFrame, pred: pd.DataFrame, q_index: int, q_slug: str):
    # choose join key
    join_key = "NCT" if ("NCT" in gt.columns and "NCT" in pred.columns) else ("PMID" if ("PMID" in gt.columns and "PMID" in pred.columns) else None)
    if not join_key:
        return {
            "question_index": q_index,
            "question_slug": q_slug,
            "rows_gt": len(gt),
            "rows_pred": len(pred),
            "rows_matched": 0,
            "target_cols": "",
            "cell_accuracy": 0.0,
            "row_accuracy": 0.0,
            "note": "No common key (NCT/PMID) to join."
        }

    # dedupe by key
    gt = gt.drop_duplicates(subset=[join_key])
    pred = pred.drop_duplicates(subset=[join_key])

    # figure target columns = overlap minus default ids
    gt_cols = [c for c in gt.columns if c not in DEFAULT_ID_COLS]
    pred_cols = [c for c in pred.columns if c not in DEFAULT_ID_COLS]
    target_cols = sorted(set(gt_cols).intersection(pred_cols))

    if not target_cols:
        return {
            "question_index": q_index,
            "question_slug": q_slug,
            "rows_gt": len(gt),
            "rows_pred": len(pred),
            "rows_matched": 0,
            "target_cols": "",
            "cell_accuracy": 0.0,
            "row_accuracy": 0.0,
            "note": "No overlapping derived columns to compare."
        }

    # align and compare
    merged = gt[[join_key] + target_cols].merge(
        pred[[join_key] + target_cols],
        on=join_key, how="inner", suffixes=("_gt","_pred")
    )
    if merged.empty:
        return {
            "question_index": q_index,
            "question_slug": q_slug,
            "rows_gt": len(gt),
            "rows_pred": len(pred),
            "rows_matched": 0,
            "target_cols": ",".join(target_cols),
            "cell_accuracy": 0.0,
            "row_accuracy": 0.0,
            "note": "No matched keys."
        }

    total_cells = 0
    correct_cells = 0
    row_all_correct = 0

    # per-row diff collection
    diffs = []

    # which columns are list-like?
    list_cols = {c for c in target_cols if _is_list_column(merged[f"{c}_gt"]) or _is_list_column(merged[f"{c}_pred"])}

    for _, r in merged.iterrows():
        row_correct = True
        rec = {join_key: r[join_key]}
        for c in target_cols:
            gv = r[f"{c}_gt"]
            pv = r[f"{c}_pred"]
            if c in list_cols:
                g_norm = _norm_listish(gv)
                p_norm = _norm_listish(pv)
                ok = set(g_norm) == set(p_norm)
                rec[f"{c}_gt"] = ", ".join(g_norm)
                rec[f"{c}_pred"] = ", ".join(p_norm)
            else:
                g_norm = _norm_scalar(gv)
                p_norm = _norm_scalar(pv)
                ok = (g_norm == p_norm)
                rec[f"{c}_gt"] = g_norm
                rec[f"{c}_pred"] = p_norm

            total_cells += 1
            correct_cells += int(ok)
            if not ok:
                row_correct = False
        row_all_correct += int(row_correct)
        if not row_correct:
            diffs.append(rec)

    # write diffs for this question (only wrong rows)
    if diffs:
        diff_df = pd.DataFrame(diffs)
        diff_df.to_csv(DIFF_DIR / f"{q_index:02d}_{q_slug}_diff.csv", index=False, encoding="utf-8")

    return {
        "question_index": q_index,
        "question_slug": q_slug,
        "rows_gt": len(gt),
        "rows_pred": len(pred),
        "rows_matched": len(merged),
        "target_cols": ",".join(target_cols),
        "cell_accuracy": (correct_cells / total_cells) if total_cells else 0.0,
        "row_accuracy": (row_all_correct / len(merged)) if len(merged) else 0.0,
        "note": ""
    }

# gather the just-written result files and evaluate 1..10
summary = []
for i in range(1, 11):
    gt_path = GT_DIR / f"q{i}_cat3_table.csv"
    # find the produced CSV for this question index (e.g., 001_*.csv)
    matches = sorted(outdir.glob(f"{i:03}_*.csv"))
    if not matches:
        summary.append({
            "question_index": i, "question_slug": f"q{i}",
            "rows_gt": 0, "rows_pred": 0, "rows_matched": 0,
            "target_cols": "", "cell_accuracy": 0.0, "row_accuracy": 0.0,
            "note": "No produced CSV found for this index."
        })
        continue

    pred_path = matches[0]
    q_slug = pred_path.stem[4:][:60]  # strip '001_' prefix
    try:
        gt_df = pd.read_csv(gt_path)
    except Exception:
        try:
            gt_df = pd.read_excel(gt_path)
        except Exception:
            summary.append({
                "question_index": i, "question_slug": q_slug,
                "rows_gt": 0, "rows_pred": 0, "rows_matched": 0,
                "target_cols": "", "cell_accuracy": 0.0, "row_accuracy": 0.0,
                "note": f"Could not read ground truth at {gt_path}"
            })
            continue

    try:
        pred_df = pd.read_csv(pred_path)
    except Exception:
        pred_df = pd.read_excel(pred_path)

    res = evaluate_pair(gt_df, pred_df, i, q_slug)
    summary.append(res)

# write summary
summary_df = pd.DataFrame(summary)
summary_df = summary_df.sort_values("question_index")
summary_csv = outdir / "eval_summary.csv"
summary_df.to_csv(summary_csv, index=False, encoding="utf-8")

# pretty print to console
print("\n=== EVALUATION SUMMARY ===")
with pd.option_context("display.max_colwidth", 120):
    print(summary_df[["question_index","question_slug","rows_gt","rows_pred","rows_matched","target_cols","cell_accuracy","row_accuracy","note"]])

print(f"\nSaved per-question diffs (where applicable) to: {DIFF_DIR}")
print(f"Saved evaluation summary to: {summary_csv}")



=== EVALUATION SUMMARY ===
   question_index question_slug  rows_gt  rows_pred  rows_matched target_cols  \
0               1            q1        0          0             0               
1               2            q2        0          0             0               
2               3            q3        0          0             0               
3               4            q4        0          0             0               
4               5            q5        0          0             0               
5               6            q6        0          0             0               
6               7            q7        0          0             0               
7               8            q8        0          0             0               
8               9            q9        0          0             0               
9              10           q10        0          0             0               

   cell_accuracy  row_accuracy                                   note  
0       